# Search intent-first website architecture: the intent mismatch audit

Companion notebook for the MLforSEO guide **"How to Build a Search Intent-First Website Architecture"** by Lazarina Stoy.

The guide argues that the defining characteristic of a high-performing page is an **intent match**, and that your architecture's job is to move visitors from one intent to the next. This notebook gives you the data side of that, from a Search Console export:

1. **Map your pages to page types** (resource, company, product) and the intent each one is *meant* to serve.
2. **Classify the queries** each page actually ranks for, with simple rule-based intent filters.
3. **Flag intent mismatches**: pages whose real query intent doesn't match their job, and pages with mixed intent.
4. **See the KPI trade-offs** per page type: clicks, impressions, CTR and position (and engagement and key events if you add GA4 data).
5. **Find intent bridges**: for each resource page, the product page that shares the most query vocabulary, which is your first candidate for a contextual link or CTA.
6. **Export** everything to CSV for Data Studio or Sheets.

No API keys needed. Runs in Google Colab or locally (Python 3.9+, pandas).

In [ ]:
# If you're running locally: pip install pandas
import re
import pandas as pd
pd.set_option('display.max_colwidth', 70)

## 1. Load your Search Console data

Export a **Performance** report with the dimensions **page and query** for the last 3 to 16 months. The Search Console API, the Search Analytics for Sheets add-on, or the BigQuery bulk export all give you this; the web interface's export does not combine page and query, so use one of those.

Columns needed: `page`, `query`, `clicks`, `impressions`, `position`. Set `USE_SAMPLE = False` to use your own file.

In [ ]:
USE_SAMPLE = True
CSV_PATH = 'gsc_page_query.csv'

if USE_SAMPLE:
    # Made-up example: a small project-management SaaS site
    rows = [
        # page, query, clicks, impressions, position
        ('/blog/what-is-a-gantt-chart', 'what is a gantt chart', 410, 12000, 3.1),
        ('/blog/what-is-a-gantt-chart', 'gantt chart examples', 150, 5200, 4.4),
        ('/blog/what-is-a-gantt-chart', 'gantt chart template', 90, 6100, 7.9),
        ('/blog/what-is-a-gantt-chart', 'gantt chart software', 25, 3900, 14.2),
        ('/blog/kanban-vs-scrum', 'kanban vs scrum', 260, 7400, 2.7),
        ('/blog/kanban-vs-scrum', 'difference between kanban and scrum', 120, 3100, 3.5),
        ('/blog/best-project-management-tools', 'best project management software', 70, 9800, 8.1),
        ('/blog/best-project-management-tools', 'project management tools comparison', 40, 2600, 6.3),
        ('/blog/best-project-management-tools', 'trello alternatives', 35, 1900, 5.0),
        ('/features/gantt-charts', 'gantt chart software', 55, 2400, 6.8),
        ('/features/gantt-charts', 'online gantt chart maker free', 30, 1700, 9.4),
        ('/features/gantt-charts', 'how to make a gantt chart', 80, 4300, 5.2),
        ('/features/gantt-charts', 'what is a gantt chart used for', 45, 2900, 8.8),
        ('/features/kanban-boards', 'kanban board software', 40, 1500, 7.2),
        ('/features/kanban-boards', 'kanban board app', 22, 900, 6.1),
        ('/pricing', 'acme pricing', 140, 600, 1.1),
        ('/pricing', 'project management software pricing', 18, 2100, 11.5),
        ('/pricing', 'acme free plan', 35, 300, 1.4),
        ('/about', 'acme company', 30, 250, 1.2),
        ('/about', 'acme careers', 12, 140, 1.8),
        ('/about', 'who owns acme', 8, 90, 1.5),
        ('/', 'acme', 900, 2200, 1.0),
        ('/', 'acme login', 300, 700, 1.0),
    ]
    gsc = pd.DataFrame(rows, columns=['page', 'query', 'clicks', 'impressions', 'position'])
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        CSV_PATH = next(iter(uploaded))
    except ImportError:
        pass
    gsc = pd.read_csv(CSV_PATH)

# Work with paths, so full URLs and paths both work
gsc['page'] = gsc['page'].str.replace(r'^https?://[^/]+', '', regex=True)
gsc.head()

## 2. Map pages to page types (and the intent each one should serve)

From the guide, generally speaking (and it is a big generalisation) there are three types of pages:

* **Resource pages** (blog, guides, glossaries): informational intent; they grow traffic and can support other goals.
* **Company pages** (home, about, team, careers): navigational intent; they build the brand and trust.
* **Product pages** (product, features, solutions, services, pricing): transactional and commercial intent; this is where conversions happen.

Edit the URL patterns to fit your site. The first pattern that matches wins.

In [ ]:
PAGE_TYPES = [
    # (page type, URL regex, intents this page type is meant to serve)
    ('product',  r'^/(features|product|products|solutions|services|pricing|plans|shop)(/|$)', {'transactional', 'commercial'}),
    ('company',  r'^/($|about|team|careers|contact|press|values)', {'navigational'}),
    ('resource', r'^/(blog|guides|learn|resources|glossary|academy)(/|$)', {'informational', 'commercial'}),
]

def page_type(path):
    for name, rx, intents in PAGE_TYPES:
        if re.search(rx, path):
            return name
    return 'other'

TARGET_INTENTS = {name: intents for name, _, intents in PAGE_TYPES}
pages = gsc[['page']].drop_duplicates().assign(page_type=lambda d: d['page'].map(page_type))
pages

## 3. Classify the queries

Simple rule-based filters, the same idea as the free Search Intent Keyword Classifier template for Data Studio. Add your brand to `BRAND_TERMS` so branded searches count as navigational. If you work in several languages, use the multilingual filters from the companion notebook for the international keyword research guide.

In [ ]:
BRAND_TERMS = ['acme']

FILTERS = {
    'navigational':  ['login', 'log in', 'sign in', 'careers', 'contact', 'customer service'],
    'transactional': ['buy', 'order', 'purchase', 'price', 'pricing', 'cost', 'free trial', 'trial', 'demo',
                      'sign up', 'signup', 'plan', 'plans', 'coupon', 'discount', 'download', 'app',
                      'software', 'tool', 'maker', 'near me'],
    'commercial':    ['best', 'top', 'review', 'reviews', 'vs', 'versus', 'compare', 'comparison',
                      'alternative', 'alternatives', 'cheap', 'affordable'],
    'informational': ['how', 'what', 'why', 'when', 'who', 'which', 'where', 'guide', 'tutorial', 'examples',
                      'ideas', 'tips', 'template', 'definition', 'meaning', 'difference', 'learn'],
}
PRIORITY = ['navigational', 'commercial', 'transactional', 'informational']

def _compile(words):
    return re.compile(r'\b(?:' + '|'.join(sorted(map(re.escape, words), key=len, reverse=True)) + r')\b', re.I)

_rx = {k: _compile(v) for k, v in FILTERS.items()}
_brand = _compile(BRAND_TERMS)

def classify(query):
    # A brand name on its own ("acme") is navigational. A brand plus a modifier
    # ("acme pricing") takes the modifier's intent; the branded column keeps the flag.
    matched = [k for k in PRIORITY if _rx[k].search(query)]
    if matched:
        return matched[0]
    return 'navigational' if _brand.search(query) else 'unclassified'

gsc['intent'] = gsc['query'].map(classify)
gsc['branded'] = gsc['query'].map(lambda q: bool(_brand.search(q)))
gsc[['page', 'query', 'clicks', 'branded', 'intent']]

## 4. Flag intent mismatches

For each page, the notebook works out the click-weighted intent mix of the queries it ranks for, and compares the dominant intent with the intents its page type should serve.

* **match**: the page mostly attracts the intent it was built for.
* **mismatch**: it mostly attracts a different intent. Either the content needs to change, or a different page should be ranking for those queries.
* **mixed**: no intent reaches 60% of clicks. Look at the SERP: Google may be showing a mixed-intent result page too, in which case serve both intents on the page (or with clear links between pages).

Use impressions instead of clicks (`WEIGHT = 'impressions'`) if you want to see the intent of the demand you *could* capture, not just what you capture today.

In [ ]:
WEIGHT = 'clicks'
MIXED_THRESHOLD = 0.60

mix = (gsc.pivot_table(index='page', columns='intent', values=WEIGHT, aggfunc='sum', fill_value=0))
share = mix.div(mix.sum(axis=1), axis=0)

audit = pages.set_index('page').join(share.round(2))
audit['dominant_intent'] = share.idxmax(axis=1)
audit['dominant_share'] = share.max(axis=1).round(2)

def verdict(row):
    targets = TARGET_INTENTS.get(row['page_type'], set())
    if row['dominant_share'] < MIXED_THRESHOLD:
        return 'mixed'
    if not targets:
        return 'no target set'
    return 'match' if row['dominant_intent'] in targets else 'mismatch'

audit['verdict'] = audit.apply(verdict, axis=1)
audit = audit.join(gsc.groupby('page')[['clicks', 'impressions']].sum())
audit.sort_values(['verdict', 'impressions'], ascending=[True, False])

In the example, `/features/gantt-charts` is a product page, but most of its clicks come from "how to make a gantt chart" and "what is a gantt chart used for". That's informational demand landing on a money page. The usual fixes: strengthen the resource page for those queries and link from it to the feature page with a CTA that matches the reader's intent, or add a short "how it works" section to the feature page if the SERP rewards that.

## 5. The KPI trade-offs per page type

The guide's point: different intents move different KPIs. Resource pages bring sessions and clicks; product pages bring fewer visits but the money-making interactions. Seeing them side by side keeps everyone honest about what each part of the site is for.

Optionally add a GA4 landing-page export (`page`, `sessions`, `engagement_rate`, `key_events`) to include engagement and conversions. In GA4, "key events" replaced "conversions" in 2024 and "engagement rate" is the inverse of bounce rate.

In [ ]:
kpis = (gsc.assign(page_type=gsc['page'].map(page_type))
           .groupby('page_type')
           .agg(pages=('page', 'nunique'), clicks=('clicks', 'sum'), impressions=('impressions', 'sum')))
kpis['ctr_%'] = (100 * kpis['clicks'] / kpis['impressions']).round(1)
pos = (gsc.assign(page_type=gsc['page'].map(page_type), w=gsc['position'] * gsc['impressions'])
          .groupby('page_type')[['w', 'impressions']].sum())
kpis['avg_position'] = (pos['w'] / pos['impressions']).round(1)

GA4_CSV = None   # e.g. 'ga4_landing_pages.csv'
if GA4_CSV:
    ga4 = pd.read_csv(GA4_CSV)
    ga4['page_type'] = ga4['page'].map(page_type)
    kpis = kpis.join(ga4.groupby('page_type').agg(sessions=('sessions', 'sum'),
                                                  key_events=('key_events', 'sum'),
                                                  engagement_rate=('engagement_rate', 'mean')))
kpis

## 6. Intent bridges: where should each resource page link to?

The architecture's job is to guide people from informational to commercial and transactional intent without sending them back to the search results. A good first candidate for a contextual link or CTA from a resource page is the product page whose queries share the most vocabulary with it.

This is a simple word-overlap score (Jaccard similarity on query words, minus stop words and your brand). It's a starting point for a human decision, not an automatic linking rule. For something more semantic, swap in sentence embeddings, as shown in other MLforSEO notebooks.

In [ ]:
STOP = set('a an and are as at be by can do does for from how i in is it of on or the to vs what when where which who why with your you used make'.split()) | set(BRAND_TERMS)

def vocab(page):
    words = ' '.join(gsc.loc[gsc['page'] == page, 'query']).lower().split()
    return {w for w in words if w not in STOP and len(w) > 2}

types = pages.set_index('page')['page_type']
resources = types[types == 'resource'].index
products = types[types == 'product'].index

bridges = []
for r in resources:
    rv = vocab(r)
    scored = []
    for p in products:
        pv = vocab(p)
        score = len(rv & pv) / len(rv | pv) if rv | pv else 0
        scored.append((score, p, sorted(rv & pv)))
    score, best, shared = max(scored) if scored else (0, None, [])
    bridges.append({'resource_page': r, 'suggested_link_to': best if score > 0 else None,
                    'overlap': round(score, 2), 'shared_terms': ', '.join(shared),
                    'resource_impressions': int(gsc.loc[gsc['page'] == r, 'impressions'].sum())})

bridges = pd.DataFrame(bridges).sort_values('resource_impressions', ascending=False)
bridges

## 7. Export

In [ ]:
gsc.to_csv('queries_with_intent.csv', index=False)
audit.reset_index().to_csv('page_intent_audit.csv', index=False)
bridges.to_csv('intent_bridges.csv', index=False)
try:
    from google.colab import files
    for f in ['queries_with_intent.csv', 'page_intent_audit.csv', 'intent_bridges.csv']:
        files.download(f)
except ImportError:
    print('Saved queries_with_intent.csv, page_intent_audit.csv and intent_bridges.csv')

## Where to go next

* [What Is Search Intent in SEO? A Practical Guide](https://www.mlforseo.com/machine-learning-implementation-guides/keyword-research/what-is-search-intent-in-seo-a-practical-guide-to-using-it-in-semantic-keyword-research/)
* [Search-Intent Classification with Machine Learning: The Approaches Explained](https://www.mlforseo.com/machine-learning-implementation-guides/serp-analysis/search-intent-classification-with-machine-learning-approaches/)
* [The Semantic ML-enabled Keyword Research course](https://academy.mlforseo.com/product/semantic-ml-enabled-keyword-research-course/) on the MLforSEO Academy